# LeetCode 3625 — Count Number of Trapezoids II bằng Python

- Link: https://leetcode.com/problems/count-number-of-trapezoids-ii/
- Độ khó: **Hard**
- Chủ đề: Hình học tổ hợp, **chuẩn hoá hệ số góc** bằng `gcd`, bảng băm nhóm đoạn thẳng, **bao hàm – loại trừ** (hình bình hành bị đếm hai lần)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countTrapezoids(self, points: List[List[int]]) -> int:
```

## 1. Mục tiêu học

1. Hình thang = tứ giác **lồi** có ít nhất một cặp cạnh song song. Một cặp đoạn thẳng **song song, nằm trên hai đường thẳng khác nhau** luôn tạo đúng một hình thang (bốn đầu mút).
2. Biểu diễn hệ số góc **chính xác** bằng cặp số nguyên tối giản `(dx, dy)` thay vì số thực.
3. Biểu diễn **đường thẳng** bằng `(hệ số góc, c)` với `c = dx·y − dy·x` (bất biến trên đường).
4. Hình bình hành có **hai** cặp cạnh song song ⇒ bị đếm hai lần ⇒ trừ số hình bình hành, đếm bằng **trung điểm chung của hai đường chéo**.
5. Kiểm chứng bằng vét cạn mọi bộ 4 điểm.

## 2. Đề bài & ví dụ

Đếm số hình thang (tập 4 điểm phân biệt tạo tứ giác lồi có ít nhất một cặp cạnh song song) chọn từ `points`.

**Ví dụ 1:** `[[-3,2],[3,0],[2,3],[3,2],[2,-3]]` → `2`.

**Ví dụ 2:** `[[0,0],[1,0],[0,1],[2,1]]` → `1`.

**Ràng buộc:** `n ≤ 500`, toạ độ trong `[−1000, 1000]`, các điểm phân biệt.

In [1]:
from typing import List
from itertools import combinations
from collections import defaultdict, Counter
from math import gcd
import random, time

examples = [([[-3, 2], [3, 0], [2, 3], [3, 2], [2, -3]], 2), ([[0, 0], [1, 0], [0, 1], [2, 1]], 1)]
for p, exp in examples:
    print(p, "-> kỳ vọng", exp)

[[-3, 2], [3, 0], [2, 3], [3, 2], [2, -3]] -> kỳ vọng 2
[[0, 0], [1, 0], [0, 1], [2, 1]] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Chuẩn hoá hệ số góc

Đoạn `P → Q`: `(dx, dy) = Q − P`, chia cho `g = gcd(|dx|, |dy|)`, rồi chuẩn dấu (ví dụ `dx > 0`, hoặc `dx = 0` thì `dy = 1`). Hai đoạn song song ⇔ cùng cặp chuẩn hoá.

In [2]:
def slope(p, q):
    dx, dy = q[0] - p[0], q[1] - p[1]; g = gcd(abs(dx), abs(dy)); dx //= g; dy //= g
    if dx < 0 or (dx == 0 and dy < 0): dx, dy = -dx, -dy
    return dx, dy

print(slope((0, 0), (2, 4)), slope((3, 3), (1, -1)), slope((0, 0), (0, -5)))

(1, 2) (1, 2) (0, 1)


### 3.2. Định danh đường thẳng

Với hướng `(dx, dy)` cố định, mọi điểm `(x, y)` trên cùng một đường thẳng có cùng `c = dx·y − dy·x` (tích có hướng với vectơ chỉ phương). Hai đoạn song song nằm trên **cùng** đường thẳng ⇔ cùng `c`.

### 3.3. Tứ giác lồi và đường chéo

Bốn điểm tạo tứ giác lồi ⇔ có cách sắp vòng sao cho mọi tích có hướng liên tiếp cùng dấu (khác 0). Hình bình hành ⇔ hai đường chéo cắt nhau tại **trung điểm chung**.

In [3]:
def cross(o, a, b):
    return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

def convex_order(pts):
    a, b, c, d = pts
    for order in ((a, b, c, d), (a, b, d, c), (a, c, b, d)):
        s = [cross(order[i], order[(i + 1) % 4], order[(i + 2) % 4]) for i in range(4)]
        if all(x > 0 for x in s) or all(x < 0 for x in s): return order
    return None

print(convex_order([(0, 0), (1, 1), (1, 0), (0, 1)]), convex_order([(0, 0), (2, 0), (1, 1), (1, 3)]))

((0, 0), (1, 0), (1, 1), (0, 1)) None


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

- Mỗi hình thang có một cặp cạnh song song = hai đoạn song song trên hai đường khác nhau. Ngược lại, hai đoạn song song trên hai đường khác nhau `AB`, `CD` luôn cho tứ giác lồi `A B D C` (hoặc `A B C D`) — một hình thang.
- Đếm cặp đoạn như vậy: với mỗi hệ số góc, có `S` đoạn chia theo đường thẳng thành các nhóm `L_1, L_2, …` ⇒ số cặp khác đường `= (S² − Σ L_i²) / 2`.
- Hình bình hành có **hai** cặp cạnh song song ⇒ bị đếm 2 lần ⇒ trừ 1 lần. Số hình bình hành = số cặp đoạn (đường chéo) **cùng trung điểm** và **không cùng hệ số góc**.

Ví dụ 2: điểm `(0,0),(1,0),(0,1),(2,1)`: đoạn `(0,0)-(1,0)` và `(0,1)-(2,1)` cùng hệ số góc `(1,0)`, khác đường ⇒ 1 hình thang; không có hình bình hành ⇒ `1`. ✔

## 5. Cách 1 — Brute Force: mọi bộ 4 điểm

In [4]:
def parallel(p, q, r, s):
    return (q[0] - p[0]) * (s[1] - r[1]) == (q[1] - p[1]) * (s[0] - r[0])

def brute(points):
    P = [tuple(p) for p in points]; cnt = 0
    for quad in combinations(P, 4):
        o = convex_order(quad)
        if o is None: continue
        a, b, c, d = o
        if parallel(a, b, d, c) or parallel(b, c, a, d): cnt += 1
    return cnt

for p, exp in examples:
    print(brute(p), "kỳ vọng", exp)

2 kỳ vọng 2
1 kỳ vọng 1


**Nút thắt:** `C(500, 4) ≈ 2,6·10^9` bộ.

## 6. Ý tưởng thuật toán / bước đột phá

Duyệt mọi cặp điểm (`C(n,2) ≈ 1,25·10^5` đoạn):

1. `by_slope[k]` = số đoạn hướng `k`; `by_line[(k, c)]` = số đoạn trên đường `(k, c)`.
2. `by_mid[m]` = số đoạn có trung điểm `m` (dùng tổng toạ độ `x₁+x₂, y₁+y₂` để tránh chia 2); `by_mid_slope[(m, k)]` = số đoạn có trung điểm `m` và hướng `k`.

`trap = Σ_k (S_k² − Σ_{line ∈ k} L²) / 2`, `para = Σ_m (M_m² − Σ_k M_{m,k}²) / 2`, đáp án `trap − para`.

## 7. Các cách tiếp theo

- **Cách 2:** liệt kê trực tiếp mọi cặp đoạn song song khác đường, lưu **tập 4 điểm** để khử trùng — đúng nhưng `O(n⁴)` trường hợp xấu.
- **Cách 3:** công thức đếm bằng bảng băm (lời giải nộp).
- **Cách 4:** như Cách 3 nhưng đếm hình bình hành bằng cách duyệt trực tiếp các cặp đường chéo cùng trung điểm (kiểm tra lại công thức trừ).

### 7.1. Cách 2 — Liệt kê cặp đoạn + khử trùng

In [5]:
def enumerate_pairs(points):
    P = [tuple(p) for p in points]; groups = defaultdict(list)
    for p, q in combinations(P, 2):
        k = slope(p, q); c = k[0] * p[1] - k[1] * p[0]
        groups[k].append((c, p, q))
    quads = set()
    for k, segs in groups.items():
        for (c1, a, b), (c2, d, e) in combinations(segs, 2):
            if c1 != c2: quads.add(frozenset((a, b, d, e)))
    return len(quads)

for p, exp in examples:
    print(enumerate_pairs(p), "kỳ vọng", exp)

2 kỳ vọng 2
1 kỳ vọng 1


### 7.2. Cách 3 — Công thức bảng băm (lời giải chính)

In [6]:
def hashing(points):
    by_slope = Counter(); by_line = Counter(); by_mid = Counter(); by_mid_slope = Counter()
    for (x1, y1), (x2, y2) in combinations(points, 2):
        k = slope((x1, y1), (x2, y2)); c = k[0] * y1 - k[1] * x1; m = (x1 + x2, y1 + y2)
        by_slope[k] += 1; by_line[(k, c)] += 1; by_mid[m] += 1; by_mid_slope[(m, k)] += 1
    trap = (sum(v * v for v in by_slope.values()) - sum(v * v for v in by_line.values())) // 2
    para = (sum(v * v for v in by_mid.values()) - sum(v * v for v in by_mid_slope.values())) // 2
    return trap - para

for p, exp in examples:
    print(hashing(p), "kỳ vọng", exp)

2 kỳ vọng 2
1 kỳ vọng 1


**Vì sao `(S² − Σ L²)/2`?** Số cặp có thứ tự của `S` đoạn là `S²`; bỏ cặp cùng đường (`Σ L²`, gồm cả cặp một đoạn với chính nó) rồi chia 2 cho cặp không thứ tự.

### 7.3. Cách 4 — Đếm hình bình hành trực tiếp

In [7]:
def count_parallelograms_direct(points):
    P = [tuple(p) for p in points]; mids = defaultdict(list)
    for p, q in combinations(P, 2): mids[(p[0] + q[0], p[1] + q[1])].append((p, q))
    cnt = 0
    for segs in mids.values():
        for (a, b), (c, d) in combinations(segs, 2):
            if slope(a, b) != slope(c, d): cnt += 1          # hai đường chéo không thẳng hàng
    return cnt

def hashing_direct_para(points):
    by_slope = Counter(); by_line = Counter()
    for (x1, y1), (x2, y2) in combinations(points, 2):
        k = slope((x1, y1), (x2, y2)); by_slope[k] += 1; by_line[(k, k[0] * y1 - k[1] * x1)] += 1
    trap = (sum(v * v for v in by_slope.values()) - sum(v * v for v in by_line.values())) // 2
    return trap - count_parallelograms_direct(points)

for p, exp in examples:
    print(hashing_direct_para(p), "kỳ vọng", exp)

2 kỳ vọng 2
1 kỳ vọng 1


## 8. TRACE từng bước

In [8]:
def trace(points):
    by_slope = defaultdict(lambda: defaultdict(int)); by_mid = defaultdict(lambda: defaultdict(int))
    for (x1, y1), (x2, y2) in combinations(points, 2):
        k = slope((x1, y1), (x2, y2)); c = k[0] * y1 - k[1] * x1
        by_slope[k][c] += 1; by_mid[(x1 + x2, y1 + y2)][k] += 1
    trap = 0
    for k, lines in by_slope.items():
        S = sum(lines.values()); t = (S * S - sum(v * v for v in lines.values())) // 2
        if t: print(f"hướng {k}: các đường {dict(lines)} -> {t} cặp song song khác đường")
        trap += t
    para = 0
    for m, sl in by_mid.items():
        M = sum(sl.values()); t = (M * M - sum(v * v for v in sl.values())) // 2
        if t: print(f"trung điểm {m[0]/2, m[1]/2}: {dict(sl)} -> {t} hình bình hành")
        para += t
    print(f"trap = {trap}, para = {para}, đáp án = {trap - para} | brute = {brute(points)}")

trace([[0, 0], [2, 0], [0, 2], [2, 2], [1, 3]])

hướng (1, 0): các đường {0: 1, 2: 1} -> 1 cặp song song khác đường
hướng (0, 1): các đường {0: 1, -2: 1} -> 1 cặp song song khác đường
hướng (1, 1): các đường {0: 1, 2: 1} -> 1 cặp song song khác đường
hướng (1, -1): các đường {2: 1, 4: 1} -> 1 cặp song song khác đường
trung điểm (1.0, 1.0): {(1, 1): 1, (1, -1): 1} -> 1 hình bình hành
trap = 4, para = 1, đáp án = 3 | brute = 3


## 9. Code nộp LeetCode

In [9]:
from math import gcd

class Solution:
    def countTrapezoids(self, points: List[List[int]]) -> int:
        by_slope = {}
        by_line = {}
        by_mid = {}
        by_mid_slope = {}
        n = len(points)
        for i in range(n):
            x1, y1 = points[i]
            for j in range(i + 1, n):
                x2, y2 = points[j]
                dx, dy = x2 - x1, y2 - y1
                g = gcd(dx, dy)
                dx //= g; dy //= g
                if dx < 0 or (dx == 0 and dy < 0):
                    dx, dy = -dx, -dy
                k = (dx, dy)
                line = (dx, dy, dx * y1 - dy * x1)
                mid = (x1 + x2, y1 + y2)
                ms = (x1 + x2, y1 + y2, dx, dy)
                by_slope[k] = by_slope.get(k, 0) + 1
                by_line[line] = by_line.get(line, 0) + 1
                by_mid[mid] = by_mid.get(mid, 0) + 1
                by_mid_slope[ms] = by_mid_slope.get(ms, 0) + 1
        sq = lambda d: sum(v * v for v in d.values())
        trap = (sq(by_slope) - sq(by_line)) // 2
        para = (sq(by_mid) - sq(by_mid_slope)) // 2
        return trap - para

sol = Solution()
for p, exp in examples:
    got = sol.countTrapezoids(p)
    print(got, "✔" if got == exp else "✘")

2 ✔
1 ✔


## 10. Vì sao thuật toán đúng

1. **Cặp cạnh song song ↔ hình thang:** cho hai đoạn `AB ∥ CD` trên hai đường khác nhau, bốn điểm phân biệt (đường khác nhau ⇒ không chung điểm) và tứ giác `A, B, D, C` (sắp `C, D` cùng chiều với `A, B`) là lồi — hai đường song song kẹp tứ giác giữa chúng. Ngược lại, mỗi hình thang có ít nhất một cặp cạnh như vậy.
2. **Số lần đếm:** hình thang không phải hình bình hành có **đúng một** cặp cạnh song song (một cặp đường chéo không bao giờ song song trong tứ giác lồi) ⇒ đếm 1 lần. Hình bình hành có **hai** cặp ⇒ đếm 2 lần.
3. **Đếm hình bình hành:** tứ giác lồi là hình bình hành ⇔ hai đường chéo có trung điểm chung. Hai đoạn cùng trung điểm, **khác hướng** luôn là hai đường chéo của một hình bình hành (không suy biến vì khác hướng); cùng hướng thì thẳng hàng ⇒ loại.
4. Đáp án `= trap − para`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n⁴)` | `O(1)` |
| 2. Liệt kê cặp đoạn | `O(n⁴)` xấu nhất | `O(n⁴)` |
| 3. Bảng băm | `O(n² log C)` (gcd) | `O(n²)` |
| 4. Hình bình hành trực tiếp | `O(n² + Σ M_m²)` | `O(n²)` |

## 12. Bộ test

In [10]:
METHODS = [brute, enumerate_pairs, hashing, hashing_direct_para, lambda p: Solution().countTrapezoids(p)]
for p, exp in examples:
    assert all(f(p) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [11]:
# 12.2 ca biên
assert Solution().countTrapezoids([[0, 0], [1, 0], [1, 1], [0, 1]]) == 1          # hình vuông: 1 (không phải 2)
assert Solution().countTrapezoids([[0, 0], [1, 0], [2, 0], [3, 0]]) == 0          # thẳng hàng
assert Solution().countTrapezoids([[0, 0], [1, 2], [3, 1], [5, 7]]) == brute([[0, 0], [1, 2], [3, 1], [5, 7]])
print("Ca biên: OK")

Ca biên: OK


In [12]:
# 12.3 ngẫu nhiên (lưới nhỏ để có nhiều song song, thẳng hàng, bình hành)
rng = random.Random(3625)
for _ in range(300):
    n = rng.randint(4, 9)
    pts = rng.sample([(x, y) for x in range(-3, 4) for y in range(-3, 4)], n)
    pts = [list(p) for p in pts]
    b = brute(pts)
    assert all(f(pts) == b for f in METHODS[1:]), pts
print("300 test ngẫu nhiên: OK")

300 test ngẫu nhiên: OK


In [13]:
# 12.4 test lớn n = 500
rng = random.Random(1)
pts = [list(p) for p in rng.sample([(x, y) for x in range(-30, 31) for y in range(-30, 31)], 500)]
t0 = time.time(); x = Solution().countTrapezoids(pts); t1 = time.time()
print("đáp án:", x, "| thời gian:", round(t1 - t0, 2), "s | khớp Cách 4:", x == hashing_direct_para(pts))

đáp án: 12210226 | thời gian: 0.24 s | khớp Cách 4: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không trừ hình bình hành

In [14]:
def wrong_no_para(points):
    by_slope = Counter(); by_line = Counter()
    for (x1, y1), (x2, y2) in combinations(points, 2):
        k = slope((x1, y1), (x2, y2)); by_slope[k] += 1; by_line[(k, k[0] * y1 - k[1] * x1)] += 1
    return (sum(v * v for v in by_slope.values()) - sum(v * v for v in by_line.values())) // 2      # SAI

print("sai:", wrong_no_para([[0, 0], [1, 0], [1, 1], [0, 1]]), "| đúng:", brute([[0, 0], [1, 0], [1, 1], [0, 1]]))

sai: 2 | đúng: 1


### Lỗi 2 — Không loại cặp đoạn **cùng đường thẳng**

In [15]:
def wrong_same_line(points):
    by_slope = Counter(); by_mid = Counter(); by_mid_slope = Counter()
    for (x1, y1), (x2, y2) in combinations(points, 2):
        k = slope((x1, y1), (x2, y2)); m = (x1 + x2, y1 + y2)
        by_slope[k] += 1; by_mid[m] += 1; by_mid_slope[(m, k)] += 1
    trap = sum(v * (v - 1) // 2 for v in by_slope.values())                            # SAI: gồm cặp thẳng hàng
    para = (sum(v * v for v in by_mid.values()) - sum(v * v for v in by_mid_slope.values())) // 2
    return trap - para

print("sai:", wrong_same_line([[0, 0], [1, 0], [2, 0], [3, 0]]), "| đúng:", brute([[0, 0], [1, 0], [2, 0], [3, 0]]))

sai: 15 | đúng: 0


### Lỗi 3 — Dùng hệ số góc số thực `dy / dx`

In [16]:
a, b = 1 / 3, 0.1 / 0.3
print("1/3 == 0.1/0.3 ?", a == b, "| sai số dấu phẩy động có thể tách hai hướng song song thành hai nhóm")
print("đứng: dx = 0 gây chia cho 0 nếu không xử lý riêng")

1/3 == 0.1/0.3 ? False | sai số dấu phẩy động có thể tách hai hướng song song thành hai nhóm
đứng: dx = 0 gây chia cho 0 nếu không xử lý riêng


### Lỗi 4 — Chuẩn hoá dấu không nhất quán

Nếu không ép `dx > 0` (hoặc `dx = 0 ⇒ dy > 0`), đoạn `P→Q` và `Q→P` cho hai khoá khác nhau `(1, 2)` và `(−1, −2)` ⇒ đếm thiếu cặp song song.

In [17]:
def bad_slope(p, q):
    dx, dy = q[0] - p[0], q[1] - p[1]; g = gcd(abs(dx), abs(dy)); return dx // g, dy // g      # SAI: không chuẩn dấu

print(bad_slope((0, 0), (1, 2)), bad_slope((3, 4), (2, 2)), "| chuẩn:", slope((0, 0), (1, 2)), slope((3, 4), (2, 2)))

(1, 2) (-1, -2) | chuẩn: (1, 2) (1, 2)


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đếm hình dạng từ 4 điểm ⇒ đếm qua **cặp đoạn** `O(n²)` với bảng băm theo đặc trưng (hướng, đường, trung điểm).
- Hệ số góc ⇒ cặp số nguyên tối giản + chuẩn dấu, không bao giờ dùng `float`.
- Đối tượng có "hai lý do" để được đếm (hình bình hành) ⇒ bao hàm – loại trừ.
- Tránh chia 2 cho trung điểm: dùng **tổng** toạ độ.

## 15. Biến thể của bài

1. **Bản I**: chỉ hình thang có cạnh **nằm ngang** — chỉ cần nhóm theo `y`.
2. **Đếm hình bình hành**: chính là `para`.
3. **Đếm hình chữ nhật**: đường chéo cùng trung điểm **và** cùng độ dài.
4. **Đếm tam giác không suy biến**: `C(n, 3) −` số bộ ba thẳng hàng (nhóm theo hướng từ mỗi điểm).

In [18]:
# Biến thể 3: số hình chữ nhật — đường chéo cùng trung điểm và cùng độ dài
def count_rectangles(points):
    groups = defaultdict(Counter)
    for (x1, y1), (x2, y2) in combinations(points, 2):
        groups[(x1 + x2, y1 + y2, (x1 - x2) ** 2 + (y1 - y2) ** 2)][slope((x1, y1), (x2, y2))] += 1
    total = 0
    for c in groups.values():
        M = sum(c.values()); total += (M * M - sum(v * v for v in c.values())) // 2
    return total

def brute_rect(points):
    P = [tuple(p) for p in points]; cnt = 0
    for quad in combinations(P, 4):
        o = convex_order(quad)
        if o is None: continue
        a, b, c, d = o
        dot = lambda p, q, r: (q[0] - p[0]) * (r[0] - p[0]) + (q[1] - p[1]) * (r[1] - p[1])
        if dot(a, b, d) == 0 and dot(b, c, a) == 0 and dot(c, d, b) == 0: cnt += 1
    return cnt

rng = random.Random(15)
for _ in range(200):
    pts = [list(p) for p in rng.sample([(x, y) for x in range(4) for y in range(4)], rng.randint(4, 9))]
    assert count_rectangles(pts) == brute_rect(pts)
print("Biến thể 3 OK; lưới 3x3:", count_rectangles([[x, y] for x in range(3) for y in range(3)]))

Biến thể 3 OK; lưới 3x3: 10


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | Mọi bộ 4 điểm | `O(n⁴)` | Kiểm chứng |
| 2. Cặp đoạn + tập | Liệt kê | `O(n⁴)` xấu | Minh hoạ song ánh |
| 3. Bảng băm | Đếm theo nhóm + trừ bình hành | `O(n²)` | **Lời giải chuẩn** |
| 4. Bình hành trực tiếp | Kiểm tra công thức trừ | `O(n² + …)` | Kiểm chứng |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** Các đỉnh của một hình lục giác đều? (Dùng `[[2,0],[1,2],[-1,2],[-2,0],[-1,-2],[1,-2]]` — lục giác đối xứng tâm.)

**Bài 2 (tính tay).** `[[0,0],[2,0],[0,2],[2,2],[1,3]]`.

**Bài 3 (điền TODO).** Viết `count_trapezoids_horizontal(points)` (bản I: cạnh song song phải nằm ngang).

**Bài 4 (tự viết lại).** Viết `count_parallelograms(points)` độc lập.

**Bài 5 (mini challenge).** Đếm số tam giác không suy biến trong `O(n² log)`.

In [19]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_trapezoids_horizontal(points):
    # TODO: nhóm theo y, mỗi hàng có C(cnt, 2) đoạn ngang; ghép hai hàng khác nhau
    pass

In [20]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def count_triangles(points):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Theo từng hướng, đếm cặp đoạn song song nằm trên hai đường khác nhau (`trap`). (2) Đếm các cặp đường chéo cắt nhau tại tâm với hướng khác nhau (`para`). (3) `trap − para = 9` (đối chiếu vét cạn ở mục 19).

**Bài 2.** (1) Hình vuông `(0,0),(2,0),(0,2),(2,2)` là 1 hình thang (hình bình hành, chỉ tính một lần). (2) Các bộ chứa `(1,3)` cần một cặp cạnh song song: trace ở mục 8 liệt kê từng hướng và số cặp song song khác đường. (3) Trừ một hình bình hành (hình vuông bị đếm hai lần) ⇒ đáp án `3`.

**Bài 3.** (1) `row[y]` = số điểm có tung độ `y`. (2) `s_y = C(row[y], 2)`. (3) `Σ_{y<y'} s_y s_{y'} = ((Σ s)² − Σ s²)/2`.

**Bài 4.** (1) Nhóm đoạn theo trung điểm. (2) Trừ cặp cùng hướng. (3) Chính là `para`.

**Bài 5.** (1) `C(n,3)` trừ số bộ ba thẳng hàng. (2) Mỗi đường thẳng có `L` đoạn ⇔ `t` điểm với `C(t,2) = L`. (3) Cộng `C(t, 3)` theo từng đường.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
hexa = [[2, 0], [1, 2], [-1, 2], [-2, 0], [-1, -2], [1, -2]]
assert Solution().countTrapezoids(hexa) == brute(hexa)
assert Solution().countTrapezoids([[0, 0], [2, 0], [0, 2], [2, 2], [1, 3]]) == brute([[0, 0], [2, 0], [0, 2], [2, 2], [1, 3]])
print("Bài 1:", Solution().countTrapezoids(hexa), "| Bài 2:", Solution().countTrapezoids([[0, 0], [2, 0], [0, 2], [2, 2], [1, 3]]))

Bài 1: 9 | Bài 2: 3


In [22]:
# Bài 3, 4
def count_trapezoids_horizontal_sol(points):
    row = Counter(y for _, y in points); s = [c * (c - 1) // 2 for c in row.values()]
    return (sum(s) ** 2 - sum(v * v for v in s)) // 2

def count_parallelograms_sol(points):
    by_mid = Counter(); by_ms = Counter()
    for (x1, y1), (x2, y2) in combinations(points, 2):
        m = (x1 + x2, y1 + y2); by_mid[m] += 1; by_ms[(m, slope((x1, y1), (x2, y2)))] += 1
    return (sum(v * v for v in by_mid.values()) - sum(v * v for v in by_ms.values())) // 2

def brute_horizontal(points):
    P = [tuple(p) for p in points]; cnt = 0
    for quad in combinations(P, 4):
        ys = Counter(y for _, y in quad)
        if sorted(ys.values()) == [2, 2]: cnt += 1
    return cnt

rng = random.Random(4)
for _ in range(200):
    pts = [list(p) for p in rng.sample([(x, y) for x in range(4) for y in range(4)], rng.randint(4, 9))]
    assert count_trapezoids_horizontal_sol(pts) == brute_horizontal(pts)
    assert count_parallelograms_sol(pts) == count_parallelograms_direct(pts)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [23]:
# Bài 5: tam giác không suy biến
def count_triangles_sol(points):
    n = len(points); lines = Counter()
    for (x1, y1), (x2, y2) in combinations(points, 2):
        k = slope((x1, y1), (x2, y2)); lines[(k, k[0] * y1 - k[1] * x1)] += 1
    col = 0
    for L in lines.values():
        t = (1 + int((1 + 8 * L) ** 0.5)) // 2               # C(t, 2) = L
        col += t * (t - 1) * (t - 2) // 6
    return n * (n - 1) * (n - 2) // 6 - col

rng = random.Random(5)
for _ in range(200):
    pts = [list(p) for p in rng.sample([(x, y) for x in range(4) for y in range(4)], rng.randint(3, 9))]
    exp = sum(1 for a, b, c in combinations(pts, 3) if cross(a, b, c) != 0)
    assert count_triangles_sol(pts) == exp
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
for mỗi cặp (p, q):
    k = (dx, dy) tối giản, chuẩn dấu; line = (k, dx*y1 - dy*x1); mid = (x1+x2, y1+y2)
    đếm by_slope[k], by_line[line], by_mid[mid], by_mid_slope[(mid, k)]
trap = (Σ by_slope² − Σ by_line²) / 2
para = (Σ by_mid² − Σ by_mid_slope²) / 2
return trap − para
```

**Checklist:**

- [ ] Hệ số góc số nguyên tối giản, chuẩn dấu?
- [ ] Loại cặp cùng đường thẳng?
- [ ] Trừ hình bình hành đúng một lần?
- [ ] Trung điểm dùng tổng toạ độ (không chia 2)?
- [ ] Đã so với vét cạn bộ 4 điểm?